# UROP: matched frozen ViT-B/16 and ResNet-50
Private comparison on the approved faculty image cohort. The two encoders read the same
RGB pixels, and the trainable heads share the same data order and optimizer settings.
Only validation selects a checkpoint. Both selections are frozen before test features.


In [ ]:
# Record values from the Kaggle GPU quota dialog and session limit.
GPU_HOURS_OBSERVED = 30
SESSION_SECONDS = 43200
MODE = 'run'  # 'preflight' verifies/times real rows and never runs held-out inference
# A previous private notebook version can be mounted for recovery after verifying its saved files.
RECOVERY = None  # absolute path to its saved urop_frozen_pair directory


In [ ]:
SOURCES = {'frozen_image_pair_v1.py': '"""Matched, pixel-only linear probes. See the registered JSON and Kaggle builder."""\nimport io\nimport json\nimport os\nimport platform\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\nimport torchvision as tv\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.data import Dataset, DataLoader\nfrom packed_image_io_v3 import ArchiveStore, sha\nfrom usable_cohort_v2 import load_input\n\nARMS = (\'vit\', \'resnet50\')\nLABELS = [\'True\', \'Satire/Parody\', \'Misleading Content\', \'Imposter Content\',\n          \'False Connection\', \'Manipulated Content\']\nFIELDS = [\'id\', \'split\', \'6_way_label\', \'source_shard\', \'bytes\', \'image_sha256\']\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef save_json(path, payload):\n    temporary = path.with_suffix(\'.tmp\')\n    temporary.write_text(json.dumps(payload, indent=2, allow_nan=False) + \'\\n\')\n    temporary.replace(path)\n\n\ndef torch_save(path, value):\n    temporary = path.with_suffix(\'.tmp\')\n    torch.save(value, temporary)\n    temporary.replace(path)\n\n\ndef load_verified(mount, cfg):\n    def unique(pattern):\n        found = list(mount.glob(pattern))\n        require(len(found) == 1, f\'Expected exactly one mounted input for {pattern}, found {len(found)}\')\n        return found[0]\n    manifest = unique(\'**/urop-faculty-usable-packed-input-v3/data/official_paired_v2/paired_manifest.csv\')\n    root = manifest.parents[2]\n    coverage = unique(\'**/urop_mm_packed_gate_20261026/packed_runtime_coverage.json\')\n    index = coverage.with_name(\'packed_image_index.sqlite\')\n    for path, key in [(manifest, \'manifest_sha256\'), (coverage, \'coverage_sha256\'), (index, \'index_sha256\')]:\n        require(sha(path) == cfg[key], f\'Unrecognized {path.name}\')\n    gate = json.loads(coverage.read_text())\n    require(gate[\'dataset_version\'] == cfg[\'dataset_version\'] and gate[\'runtime_image_bytes_verified\'] and gate[\'image_gate_passed\']\n            and gate[\'input_layout\'] == \'kaggle-packed-bin\'\n            and gate[\'cohort_id\'] == cfg[\'cohort\'] and gate[\'split_counts\'] == cfg[\'counts\']\n            and gate[\'packed_index_sha256\'] == cfg[\'index_sha256\']\n            and gate[\'manifest_sha256\'] == cfg[\'manifest_sha256\'], \'Invalid packed quality-check receipt\')\n    expected, reg = load_input(root)\n    paired = pd.read_csv(manifest, usecols=FIELDS, dtype={\'id\': str}, keep_default_na=False)\n    require(paired.id.is_unique and set(paired.id) == set(expected), \'Wrong paired cohort\')\n    canonical = pd.DataFrame.from_records(list(expected.values()), columns=FIELDS[:3])\n    paired = paired.set_index(\'id\').loc[canonical.id].reset_index()\n    for key in FIELDS[:3]:\n        require(np.array_equal(paired[key], canonical[key]), f\'Mismatch in {key}\')\n    frames = {split: paired.loc[paired.split.eq(split), FIELDS].reset_index(drop=True)\n              for split in cfg[\'counts\']}\n    for split, frame in frames.items():\n        require(len(frame) == cfg[\'counts\'][split] and set(frame[\'6_way_label\']) == set(range(6)),\n                f\'Wrong approved counts/classes for {split}\')\n    return root, index, frames, reg\n\n\ndef stage(root, index, digest):\n    # The verified loader stages exact archive bytes in /tmp and caches SQLite separately.\n    began = time.monotonic()\n    store = ArchiveStore(root, index, digest)\n    return store, time.monotonic() - began\n\n\ndef sampled(frame, n, seed):\n    require(6 <= n <= len(frame), \'Invalid stratified sample size\')\n    if n == len(frame):\n        return frame\n    buckets = [np.flatnonzero(frame[\'6_way_label\'].to_numpy() == label) for label in range(6)]\n    counts = np.asarray([len(x) for x in buckets])\n    require(np.all(counts > 0), \'Missing class in source split\')\n    weights = (n - 6) * (counts - 1) / (len(frame) - 6)\n    quotas = np.floor(weights).astype(int) + 1\n    for label in np.argsort(-(weights - np.floor(weights)), kind=\'stable\'):\n        if quotas.sum() == n:\n            break\n        if quotas[label] < counts[label]:\n            quotas[label] += 1\n    require(quotas.sum() == n and (quotas <= counts).all(), \'Stratified allocation failed\')\n    rng = np.random.default_rng(seed)\n    positions = np.sort(np.concatenate([rng.permutation(group)[:k] for group, k in zip(buckets, quotas)]))\n    return frame.iloc[positions].reset_index(drop=True)\n\n\ndef transform(cfg):\n    t = tv.transforms\n    return t.Compose([t.Resize(cfg[\'resize_shorter\'], interpolation=t.InterpolationMode.BILINEAR,\n                               antialias=cfg[\'antialias\']), t.CenterCrop(cfg[\'crop\']), t.ToTensor(),\n                      t.Normalize(cfg[\'mean\'], cfg[\'std\'])])\n\n\nclass Pixels(Dataset):\n    def __init__(self, frame, store, cfg):\n        self.rows = frame.to_dict(\'records\')\n        self.store = store\n        self.apply = transform(cfg)\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, i):\n        row = self.rows[i]\n        with Image.open(io.BytesIO(self.store.read(row))) as image:\n            image.load()\n            pixels = self.apply(image.convert(\'RGB\'))\n        return pixels, row[\'id\'], row[\'6_way_label\']\n\n\ndef load_pixels(frame, store, cfg, device):\n    return DataLoader(Pixels(frame, store, cfg), batch_size=cfg[\'feature_batch\'], shuffle=False,\n                      num_workers=cfg[\'workers\'], pin_memory=device.type == \'cuda\')\n\n\ndef backbone(arm, device):\n    if arm == \'vit\':\n        enum = tv.models.ViT_B_16_Weights.IMAGENET1K_V1\n        model = tv.models.vit_b_16(weights=enum)\n        dim = model.hidden_dim\n        model.heads = nn.Identity()\n    else:\n        require(arm == \'resnet50\', \'Unknown encoder\')\n        enum = tv.models.ResNet50_Weights.IMAGENET1K_V2\n        model = tv.models.resnet50(weights=enum)\n        dim = model.fc.in_features\n        model.fc = nn.Identity()\n    digest = sha(Path(torch.hub.get_dir()) / \'checkpoints\' / enum.url.rsplit(\'/\', 1)[-1])\n    require(digest.startswith(enum.url.rsplit(\'-\', 1)[-1].split(\'.\')[0]), \'Pretrained weight hash prefix mismatch\')\n    return model.requires_grad_(False).eval().to(device), dim, {\'enum\': str(enum), \'url\': enum.url, \'sha256\': digest}\n\n\ndef sync(device):\n    if device.type == \'cuda\':\n        torch.cuda.synchronize(device)\n\n\n@torch.inference_mode()\ndef features(frame, store, models, cfg, device):\n    values = {arm: np.empty((len(frame), spec[1]), np.float32) for arm, spec in models.items()}\n    elapsed = dict.fromkeys(models, 0.0)\n    offset, read_seconds = 0, 0.0\n    loader = iter(load_pixels(frame, store, cfg, device))\n    while offset < len(frame):\n        start = time.monotonic()\n        x, ids, labels = next(loader)\n        y = labels.numpy()\n        x = x.to(device, non_blocking=True)\n        sync(device)\n        read_seconds += time.monotonic() - start\n        require(tuple(x.shape) == (len(y), 3, cfg[\'crop\'], cfg[\'crop\']), \'Unexpected image shape\')\n        target = frame.iloc[offset:offset + len(y)]\n        require(list(ids) == target.id.tolist() and np.array_equal(y, target[\'6_way_label\']),\n                \'Feature extraction order or labels changed\')\n        for arm, (encoder, dim, _) in models.items():\n            start = time.monotonic()\n            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == \'cuda\'):\n                z = encoder(x)\n            z = z.float().cpu().numpy()\n            elapsed[arm] += time.monotonic() - start\n            require(z.shape == (len(y), dim) and np.isfinite(z).all(), f\'Invalid {arm} features\')\n            values[arm][offset:offset + len(y)] = z\n        offset += len(y)\n    return values, {\'read_seconds\': read_seconds, \'encoder_seconds\': elapsed, \'rows\': offset}\n\n\ndef head(dim, cfg, device):\n    return nn.Sequential(nn.Dropout(cfg[\'dropout\']), nn.Linear(dim, 6)).to(device)\n\n\ndef batch_order(epoch, size, seed):\n    return np.random.default_rng(np.random.SeedSequence([seed, epoch])).permutation(size)\n\n\n@torch.inference_mode()\ndef infer(model, matrix, labels, cfg, device):\n    model.eval()\n    result, loss = [], []\n    for a in range(0, len(labels), cfg[\'head_batch\']):\n        b = a + cfg[\'head_batch\']\n        x = torch.as_tensor(matrix[a:b], device=device)\n        y = torch.as_tensor(labels[a:b], dtype=torch.long, device=device)\n        logits = model(x)\n        require(tuple(logits.shape) == (len(y), 6) and torch.isfinite(logits).all(), \'Invalid logits\')\n        result.append(logits.softmax(1).cpu().numpy())\n        loss.append(F.cross_entropy(logits, y, reduction=\'none\').cpu().numpy())\n    return np.concatenate(result), np.concatenate(loss)\n\n\ndef smoke(frame, store, models, cfg, device, folder):\n    samples = {}\n    for i, split in enumerate((\'train\', \'validation\')):\n        f = sampled(frame[split], 48, cfg[\'seed\'] + i)\n        samples[split] = f, features(f, store, models, cfg, device)[0]\n    train, train_features = samples[\'train\']\n    _, val_features = samples[\'validation\']\n    proof = {}\n    for arm in models:\n        torch.manual_seed(cfg[\'seed\'])\n        m = head(models[arm][1], cfg, device)\n        opt = torch.optim.AdamW(m.parameters(), lr=cfg[\'learning_rate\'], weight_decay=cfg[\'weight_decay\'])\n        before = m[1].weight.detach().clone()\n        logits = m(torch.as_tensor(train_features[arm], device=device))\n        loss = F.cross_entropy(logits, torch.as_tensor(train[\'6_way_label\'].to_numpy(), device=device))\n        opt.zero_grad(set_to_none=True)\n        loss.backward()\n        opt.step()\n        require(logits.shape == (48, 6) and torch.isfinite(loss) and not torch.equal(before, m[1].weight),\n                \'Smoke head did not update with valid six-class logits\')\n        path = folder / f\'smoke_{arm}.pt\'\n        torch_save(path, m.state_dict())\n        reload_check(path, m, val_features[arm], cfg, device)\n        proof[arm] = {\'feature_dim\': models[arm][1], \'logits\': [48, 6], \'head_updated\': True,\n                      \'validation_reload\': True, \'loss\': float(loss.detach().cpu())}\n    return proof\n\n\n@torch.inference_mode()\ndef reload_check(path, model, data, cfg, device):\n    fresh = head(data.shape[1], cfg, device).eval()\n    fresh.load_state_dict(torch.load(path, weights_only=True, map_location=device))\n    x = torch.as_tensor(data[:32], device=device)\n    torch.testing.assert_close(fresh(x), model.eval()(x), rtol=0, atol=0)\n\n\ndef fit_scope(cfg, timing, staging, setup, quota_hours, session_seconds, free_bytes):\n    # Decode overlaps GPU work; use the observed end-to-end time, then pad.\n    rows_per_second = timing[\'rows\'] / timing[\'wall_seconds\']\n    per_row = 1.35 / rows_per_second + 0.002  # cached head passes + CSV/metrics\n    reserve = max(1200, session_seconds * .08)\n    gross = min(quota_hours * 3600, session_seconds)\n    remaining = gross - setup - reserve\n    # Allow cache files, save/verify working space and bounded overhead.\n    row_cap = min(int(remaining / per_row), max(0, (free_bytes - 2 * 1024**3) // 12500))\n    total = sum(cfg[\'counts\'].values())\n    counts = cfg[\'counts\'].copy()\n    if row_cap < total:\n        ev = counts[\'validation\'] + counts[\'test\']\n        minimum_train = cfg[\'preferred_pilot_train\']\n        if row_cap >= ev + minimum_train:\n            counts[\'train\'] = min(counts[\'train\'], row_cap - ev)\n        else:\n            counts[\'train\'] = min(minimum_train, row_cap // 2)\n            left = row_cap - counts[\'train\']\n            counts[\'validation\'] = min(counts[\'validation\'], left // 2)\n            counts[\'test\'] = min(counts[\'test\'], left - counts[\'validation\'])\n        require(min(counts.values()) >= 60, \'Measured quota/scratch does not support a six-class pilot\')\n    return {\'mode\': \'full\' if counts == cfg[\'counts\'] else \'pilot\', \'counts\': counts,\n            \'quota_hours_observed\': quota_hours, \'session_seconds\': session_seconds,\n            \'staging_seconds\': staging, \'setup_elapsed_seconds\': setup, \'saving_reserve_seconds\': reserve,\n            \'estimated_remaining_seconds\': sum(counts.values()) * per_row,\n            \'estimated_total_seconds\': setup + sum(counts.values()) * per_row + reserve,\n            \'observed_rows_per_second\': rows_per_second, \'working_free_bytes\': free_bytes}\n\n\ndef cached_features(split, frame, store, models, cfg, device, out, stop):\n    folder = out / \'cache\' / split\n    folder.mkdir(parents=True, exist_ok=True)\n    matrices = {arm: [] for arm in models}\n    for start in range(0, len(frame), 8192):\n        block = frame.iloc[start:start + 8192].reset_index(drop=True)\n        path = folder / f\'{start:07d}.npz\'\n        if not path.exists():\n            if time.monotonic() >= stop:\n                raise TimeoutError(\'Feature extraction paused at a saved shard boundary\')\n            vals, speed = features(block, store, models, cfg, device)\n            temp = path.with_suffix(\'.tmp\')\n            with temp.open(\'wb\') as f:\n                np.savez(f, ids=block.id.to_numpy(dtype=str),\n                         labels=block[\'6_way_label\'].to_numpy(), **vals)\n            temp.replace(path)\n            print(split, start + len(block), \'/\', len(frame), speed, flush=True)\n        with np.load(path, allow_pickle=False) as saved:\n            require(np.array_equal(saved[\'ids\'], block.id)\n                    and np.array_equal(saved[\'labels\'], block[\'6_way_label\']),\n                    \'Saved feature IDs or labels differ from registered manifest\')\n            for arm in models:\n                a = saved[arm]\n                require(a.shape == (len(block), models[arm][1]) and np.isfinite(a).all(),\n                        \'Invalid saved feature shard\')\n                matrices[arm].append(a)\n    return {arm: np.concatenate(chunks) for arm, chunks in matrices.items()}\n\n\ndef metrics_from_arrays(y, predicted, ce):\n    # Compute from a confusion matrix. The separate saved verifier uses sklearn.\n    cm = np.bincount(6 * y + predicted, minlength=36).reshape(6, 6)\n    total = cm.sum()\n    support, called, correct = cm.sum(1), cm.sum(0), cm.diagonal()\n    def ratio(n, d):\n        return np.divide(n, d, out=np.zeros_like(n, dtype=float), where=d != 0)\n    precision, recall, f1 = ratio(correct, called), ratio(correct, support), ratio(2*correct, called + support)\n    accuracy = float(correct.sum() / total)\n    metrics = {\'accuracy\': accuracy, \'micro_f1\': accuracy, \'macro_f1\': float(f1.mean()),\n               \'weighted_f1\': float(f1 @ support / total), \'balanced_accuracy\': float(recall[support>0].mean()),\n               \'loss\': float(np.asarray(ce, np.float64).mean()), \'samples\': int(total)}\n    report = {label: {\'precision\': float(precision[i]), \'recall\': float(recall[i]),\n                      \'f1-score\': float(f1[i]), \'support\': int(support[i])}\n              for i, label in enumerate(LABELS)}\n    for name, w in [(\'macro avg\', np.ones(6)/6), (\'weighted avg\', support/total)]:\n        report[name] = {\'precision\': float(w @ precision), \'recall\': float(w @ recall),\n                        \'f1-score\': float(w @ f1), \'support\': int(total)}\n    report[\'accuracy\'] = accuracy\n    normalized = np.divide(cm, support[:, None], out=np.zeros_like(cm, dtype=float), where=support[:, None]!=0)\n    return metrics, report, {\'labels\': LABELS, \'raw\': cm.tolist(), \'row_normalized\': normalized.tolist()}\n\n\ndef export_eval(out, prefix, frame, prob, ce):\n    y = frame[\'6_way_label\'].to_numpy(dtype=int)\n    p = prob.argmax(1)\n    arrays = metrics_from_arrays(y, p, ce)\n    table = pd.DataFrame({\'id\': frame.id, \'split\': frame.split, \'true_label\': y,\n                          \'predicted_label\': p, \'cross_entropy\': ce,\n                          **{f\'p{i}\': prob[:, i].astype(np.float64) for i in range(6)}})\n    path = out / f\'{prefix}_predictions.csv\'\n    temporary = path.with_suffix(\'.tmp\')\n    table.to_csv(temporary, index=False, float_format=\'%.17g\')\n    temporary.replace(path)\n    for suffix, obj in zip((\'metrics\', \'classification_report\', \'confusions\'), arrays):\n        save_json(out / f\'{prefix}_{suffix}.json\', obj)\n    return arrays[0]\n\n\ndef fit(arm, training, validation, frames, cfg, device, folder, stop, contract):\n    folder.mkdir(exist_ok=True)\n    y = frames[\'train\'][\'6_way_label\'].to_numpy(dtype=np.int64)\n    vy = frames[\'validation\'][\'6_way_label\'].to_numpy(dtype=np.int64)\n    torch.manual_seed(cfg[\'seed\'])\n    m = head(training.shape[1], cfg, device)\n    opt = torch.optim.AdamW(m.parameters(), lr=cfg[\'learning_rate\'], weight_decay=cfg[\'weight_decay\'])\n    state_path = folder / \'resume.pt\'\n    history = []\n    if state_path.exists():\n        state = torch.load(state_path, weights_only=True, map_location=\'cpu\')\n        require(state[\'contract\'] == contract, \'Recovery does not match frozen run\')\n        m.load_state_dict(state[\'head\'])\n        opt.load_state_dict(state[\'optimizer\'])\n        history = state[\'history\']\n    for epoch in range(len(history) + 1, cfg[\'epochs\'] + 1):\n        if time.monotonic() >= stop:\n            raise TimeoutError(\'Head training paused at a saved epoch boundary\')\n        m.train()\n        torch.manual_seed(cfg[\'seed\'] + epoch)\n        order = batch_order(epoch, len(y), cfg[\'seed\'])\n        batch = cfg[\'head_batch\']\n        summed = torch.zeros((), device=device)\n        for start in range(0, len(y), batch):\n            ids = order[start:start + batch]\n            xb, yb = torch.as_tensor(training[ids], device=device), torch.as_tensor(y[ids], device=device)\n            opt.zero_grad(set_to_none=True)\n            loss = F.cross_entropy(m(xb), yb)\n            loss.backward()\n            opt.step()\n            summed += loss.detach() * len(ids)\n        require(torch.isfinite(summed), \'Non-finite head loss\')\n        prob, ce = infer(m, validation, vy, cfg, device)\n        metrics = export_eval(folder, f\'validation_epoch{epoch}\', frames[\'validation\'], prob, ce)\n        epoch_path = folder / f\'epoch{epoch}.pt\'\n        torch_save(epoch_path, m.state_dict())\n        reload_check(epoch_path, m, validation, cfg, device)\n        history.append({\'epoch\': epoch, \'train_loss\': float(summed.item()/len(y)), \'validation\': metrics,\n                        \'weight_sha256\': sha(epoch_path)})\n        torch_save(state_path, {\'contract\': contract, \'head\': m.state_dict(), \'optimizer\': opt.state_dict(),\n                               \'history\': history})\n        save_json(folder / \'history.json\', history)\n        print(arm, history[-1], flush=True)\n    save_json(folder / \'history.json\', history)\n    best = max(history, key=lambda x: (x[\'validation\'][\'macro_f1\'], -x[\'validation\'][\'loss\'], -x[\'epoch\']))\n    source = folder / f"epoch{best[\'epoch\']}.pt"\n    require(sha(source) == best[\'weight_sha256\'], \'Selected epoch checkpoint changed\')\n    selected = folder / \'selected.pt\'\n    if selected.exists():\n        require(sha(selected) == sha(source), \'Existing selected checkpoint differs\')\n    else:\n        shutil.copyfile(source, selected)\n    m.load_state_dict(torch.load(selected, weights_only=True, map_location=device))\n    reload_check(selected, m, validation, cfg, device)\n    return m, best\n\n\ndef run(mount, out, cfg, provenance, quota_hours, session_seconds, mode):\n    from verify_frozen_image_pair_v1 import verify_run\n    began = time.monotonic()\n    require(mode in (\'preflight\', \'run\'), \'Choose a supported stage\')\n    require(torch.cuda.is_available(), \'Kaggle CUDA GPU is required\')\n    require(0 < quota_hours <= 30 and 0 < session_seconds <= 43200, \'Invalid observed Kaggle budget\')\n    out.mkdir(parents=True, exist_ok=True)\n    event = \'verified_input_hashes\'\n    try:\n        root, index, frames, registration = load_verified(mount, cfg)\n        free = {p: shutil.disk_usage(p).free for p in (\'/tmp\', \'/kaggle/working\')}\n        runtime = {\'python\': platform.python_version(), \'torch\': str(torch.__version__),\n                   \'torchvision\': tv.__version__, \'cuda\': torch.version.cuda,\n                   \'devices\': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],\n                   \'scratch_free_before_staging\': free}\n        print(event, runtime, flush=True)\n        event = \'stage_input\'\n        store, staging = stage(root, index, cfg[\'index_sha256\'])\n        device = torch.device(\'cuda:0\')\n        event = \'initialize_encoders\'\n        models = {arm: backbone(arm, device) for arm in ARMS}\n        refs = {arm: spec[2] for arm, spec in models.items()}\n        event = \'smoke\'\n        proof = smoke(frames, store, models, cfg, device, out)\n        print(\'smoke_passed\', proof, flush=True)\n        event = \'benchmark\'\n        bench = sampled(frames[\'train\'], cfg[\'benchmark_rows\'], cfg[\'seed\'])\n        t = time.monotonic()\n        _, timing = features(bench, store, models, cfg, device)\n        timing[\'wall_seconds\'] = time.monotonic() - t\n        scope = fit_scope(cfg, timing, staging, time.monotonic()-began, quota_hours, session_seconds,\n                          shutil.disk_usage(out).free)\n        evidence = {\'runtime\': runtime, \'timing\': timing, \'smoke\': proof, \'scope\': scope, \'backbones\': refs,\n                    \'cohort_registration\': registration[\'cohort_id\']}\n        save_json(out / \'preflight.json\', evidence)\n        print(\'preflight_ready\', evidence, flush=True)\n        if mode == \'preflight\':\n            return\n        event = \'register_run\'\n        resolved = out / \'resolved.json\'\n        if resolved.exists():\n            registered = json.loads(resolved.read_text())\n            require(registered[\'config\'] == cfg and registered[\'provenance\'] == provenance\n                    and registered[\'backbones\'] == refs, \'Recovery inputs, source or backbones changed\')\n            scope_counts = registered[\'scope\'][\'counts\']\n        else:\n            scope_counts = scope[\'counts\']\n        frames = {s: sampled(f, scope_counts[s], cfg[\'seed\'] + i) for i, (s, f) in enumerate(frames.items())}\n        manifest_hashes = {}\n        for split, frame in frames.items():\n            path = out / f\'{split}_manifest.csv\'\n            payload = frame[[\'id\', \'split\', \'6_way_label\', \'image_sha256\']].to_csv(index=False).encode()\n            if path.exists():\n                require(path.read_bytes() == payload, \'Recovery split manifest changed\')\n            else:\n                path.write_bytes(payload)\n            manifest_hashes[split] = sha(path)\n        if not resolved.exists():\n            save_json(resolved, {**evidence, \'config\': cfg, \'provenance\': provenance, \'manifests\': manifest_hashes})\n        else:\n            require(registered[\'manifests\'] == manifest_hashes, \'Recovery cohort changed\')\n        if (out / \'independent_verification.json\').exists():\n            verify_run(out)\n            print(\'Saved result already complete and reverified\', flush=True)\n            return\n        stop = began + min(quota_hours*3600, session_seconds) - scope[\'saving_reserve_seconds\']\n        event = \'extract_train_validation\'\n        cached = {s: cached_features(s, frames[s], store, models, cfg, device, out, stop)\n                  for s in (\'train\', \'validation\')}\n        event = \'train_validate\'\n        heads, selected = {}, {}\n        for arm in ARMS:\n            heads[arm], selected[arm] = fit(arm, cached[\'train\'][arm], cached[\'validation\'][arm], frames,\n                                           cfg, device, out / arm, stop, sha(resolved))\n        del cached\n        event = \'freeze\'\n        freeze = out / \'selection_frozen.json\'\n        freeze_record = {\'resolved_sha256\': sha(resolved), \'selected\': selected, \'backbones\': refs,\n                         \'test_manifest_sha256\': manifest_hashes[\'test\']}\n        if freeze.exists():\n            require(json.loads(freeze.read_text()) == freeze_record, \'Frozen selection changed\')\n        else:\n            save_json(freeze, freeze_record)\n        print(\'selection_frozen\', sha(freeze), flush=True)\n        event = \'test\'\n        test = frames[\'test\']\n        distinct_mask = ~test.image_sha256.isin(set(frames[\'train\'].image_sha256))\n        require(distinct_mask.any(), \'No non-overlapping test images\')\n        if scope_counts == cfg[\'counts\']:\n            require(distinct_mask.sum() == 56052, \'Full cohort exact-byte overlap mismatch\')\n        needs_test = [arm for arm in ARMS if not (out / arm / \'test_saved.json\').exists()]\n        if needs_test:\n            cache_test = cached_features(\'test\', test, store, models, cfg, device, out, stop)\n            for arm in needs_test:\n                if time.monotonic() >= stop:\n                    raise TimeoutError(\'Test paused before inference\')\n                prob, ce = infer(heads[arm], cache_test[arm], test[\'6_way_label\'].to_numpy(), cfg, device)\n                export_eval(out / arm, \'test\', test, prob, ce)\n                export_eval(out / arm, \'test_no_train_hash\', test[distinct_mask].reset_index(drop=True),\n                            prob[distinct_mask], ce[distinct_mask])\n                save_json(out / arm / \'test_saved.json\', {\'freeze_sha256\': sha(freeze),\n                          \'test_predictions_sha256\': sha(out / arm / \'test_predictions.csv\')})\n        event = \'saved_output_verification\'\n        save_json(out / \'runtime.json\', {\'current_session_seconds\': time.monotonic() - began})\n        verification = verify_run(out)\n        save_json(out / \'independent_verification.json\', verification)\n        save_json(out / \'status.json\', {\'state\': \'complete_verified\', \'event\': event})\n        from verify_frozen_image_pair_v1 import report\n        (out / \'REPORT.md\').write_text(report(out, verification))\n        print(\'complete_verified\', verification[\'differences\'], flush=True)\n    except TimeoutError as exc:\n        save_json(out / \'status.json\', {\'state\': \'needs_saved_recovery\', \'event\': event, \'reason\': str(exc)})\n        print(\'needs_saved_recovery\', event, str(exc), flush=True)\n    except Exception as exc:\n        save_json(out / \'status.json\', {\'state\': \'failed\', \'event\': event, \'reason\': str(exc)})\n        raise\n', 'image_only_vit_resnet_frozen_v1.json': '{\n  "protocol": "OFFICIAL-6W-IMAGE-FROZEN-VIT-RESNET-v1",\n  "cohort": "FACULTY-6W-USABLE-682170-v1",\n  "counts": {"train": 563601, "validation": 59299, "test": 59270},\n  "dataset_version": 1,\n  "dataset_version_id": 21277535,\n  "quality_check_script_version_id": 362643633,\n  "manifest_sha256": "96c84e9453401cc4a24999c41cdbeb182dafd08771383cf1c75feea9c3001caf",\n  "coverage_sha256": "afe8c723f91fe74f791908e0c16fab94d919af64c7cb7c9b5a37e676361d95e5",\n  "index_sha256": "fc8dc03a4f773da5762891c8392375add5fb8087251b527f3b37a880308e6018",\n  "backbones": {"vit": "ViT_B_16_Weights.IMAGENET1K_V1", "resnet50": "ResNet50_Weights.IMAGENET1K_V2"},\n  "resize_shorter": 256,\n  "crop": 224,\n  "interpolation": "bilinear",\n  "antialias": true,\n  "mean": [0.485, 0.456, 0.406],\n  "std": [0.229, 0.224, 0.225],\n  "encoder_precision": "cuda_fp16_autocast",\n  "cache_precision": "float32",\n  "seed": 42,\n  "epochs": 4,\n  "dropout": 0.2,\n  "head_batch": 256,\n  "feature_batch": 64,\n  "workers": 4,\n  "learning_rate": 0.001,\n  "weight_decay": 0.0001,\n  "preferred_pilot_train": 30000,\n  "benchmark_rows": 2048,\n  "checkpoint_selection": ["highest_validation_macro_f1", "lowest_validation_cross_entropy", "earliest_epoch"],\n  "historical_resnet50": {"accuracy": 0.8115572802429559, "macro_f1": 0.6617958357324856}\n}\n', 'packed_image_io_v3.py': '"""Lossless TAR image storage adapter. Does not change pixels or research rows.\n\nBuild a bounded index while strictly hashing/decoding every member. Read only\nthe pinned byte range during training. No image extraction or model execution.\n"""\nimport hashlib\nimport io\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport sqlite3\nimport tarfile\nimport shutil\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom collections import OrderedDict\nfrom PIL import Image\n\n_INDEX_CACHE = {}\n_ARCHIVE_CACHE = {}\n\n\ndef cache_raw_archives(root, proofs, cache, minimum_free=120 * 1024**3):\n    """Copy exact source archives into bounded cloud scratch; never re-encode.\n\n    A complete SHA proof, not a filename or prior receipt, authorizes reuse.\n    Originals remain read-only. Temporary writes are owned by this cache only.\n    """\n    root, cache = Path(root).resolve(), Path(cache).resolve()\n    total = sum(size for size, digest in proofs.values())\n    if not proofs or total > 100 * 1024**3:\n        raise ValueError(\'Raw archive cache exceeds registered storage bound\')\n    cache.mkdir(parents=True, exist_ok=True)\n    if shutil.disk_usage(cache).free < minimum_free:\n        raise ValueError(\'Insufficient cloud scratch for verified raw archive cache\')\n    started = time.monotonic()\n\n    def copy_one(item):\n        name, (size, digest) = item\n        source, target = root / name, cache / name\n        if source.parent != root or target.parent != cache or not name.endswith(\'.bin\'):\n            raise ValueError(\'Unsafe raw archive cache name\')\n        if source.stat().st_size != size:\n            raise ValueError(\'Source raw archive size changed\')\n        if target.exists():\n            if target.stat().st_size != size or sha(target) != digest:\n                raise ValueError(\'Preserve conflicting cloud raw archive cache\')\n            return {\'file\': name, \'bytes\': size, \'sha256\': digest, \'reused\': True}\n        temporary = target.with_suffix(\'.copy-\' + str(os.getpid()))\n        if temporary.exists():\n            raise ValueError(\'Preserve interrupted raw archive copy for inspection\')\n        h, count = hashlib.sha256(), 0\n        with source.open(\'rb\') as src, temporary.open(\'xb\') as dst:\n            while block := src.read(16 * 1024**2):\n                if time.monotonic() - started > 3600:\n                    raise TimeoutError(\'Raw archive staging exceeded one-hour bound\')\n                dst.write(block); h.update(block); count += len(block)\n            dst.flush(); os.fsync(dst.fileno())\n        if count != size or h.hexdigest() != digest:\n            raise ValueError(\'Raw archive cache copy SHA mismatch\')\n        os.replace(temporary, target)\n        print(json.dumps({\'state\': \'raw_archive_cached\', \'file\': name,\n                          \'bytes\': count, \'seconds\': time.monotonic()-started}), flush=True)\n        return {\'file\': name, \'bytes\': count, \'sha256\': digest, \'reused\': False}\n\n    with ThreadPoolExecutor(max_workers=4) as pool:\n        copied = list(pool.map(copy_one, sorted(proofs.items())))\n    receipt = {\'state\': \'exact_raw_archive_cache_verified\', \'source\': str(root),\n               \'cache\': str(cache), \'bytes\': total, \'archives\': copied,\n               \'seconds\': time.monotonic()-started, \'pixels_changed\': False}\n    (cache / \'raw_archive_cache_receipt.json\').write_text(json.dumps(receipt, indent=2)+\'\\n\')\n    return cache\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(4 * 1024**2), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef packed_name(archive):\n    p = PurePosixPath(archive)\n    if len(p.parts) != 1 or not archive.endswith(\'.tar\'):\n        raise ValueError(\'Expected a simple original TAR filename\')\n    return archive[:-4] + \'.bin\'\n\n\nclass HashReader:\n    def __init__(self, stream):\n        self.stream, self.digest, self.count = stream, hashlib.sha256(), 0\n\n    def read(self, size=-1):\n        block = self.stream.read(size)\n        self.digest.update(block)\n        self.count += len(block)\n        return block\n\n\ndef index_archive(root, archive, proof, rows, connection):\n    """One transaction per archive; caller commits a final index only after all."""\n    path = Path(root) / packed_name(archive)\n    if path.stat().st_size != proof[\'bytes\']:\n        raise ValueError(\'Packed archive size mismatch\')\n    expected = {r[\'id\']: r for r in rows}\n    if len(expected) != len(rows):\n        raise ValueError(\'Duplicate expected image IDs\')\n    seen = set()\n    with path.open(\'rb\') as source:\n        reader = HashReader(source)\n        with tarfile.open(fileobj=reader, mode=\'r|\') as container:\n            for member in container:\n                p = PurePosixPath(member.name)\n                if not member.isfile() or p.is_absolute() or \'..\' in p.parts:\n                    raise ValueError(\'Unexpected non-image or unsafe TAR member\')\n                ident = p.stem\n                row = expected.get(ident)\n                if row is None or ident in seen or row[\'source_shard\'] != archive:\n                    raise ValueError(\'Unexpected/duplicate image membership\')\n                if str(PurePosixPath(Path(archive).stem) / p) != row[\'image_path\']:\n                    raise ValueError(\'TAR member differs from original paired manifest\')\n                if member.size != int(row[\'bytes\']):\n                    raise ValueError(\'TAR image size differs from original paired manifest\')\n                raw = container.extractfile(member).read()\n                if len(raw) != member.size or hashlib.sha256(raw).hexdigest() != row[\'image_sha256\']:\n                    raise ValueError(\'Packed image SHA mismatch: \' + ident)\n                with Image.open(io.BytesIO(raw)) as image:\n                    image.load()\n                    image.convert(\'RGB\').load()\n                connection.execute(\'INSERT INTO images VALUES (?,?,?,?,?)\',\n                                   (ident, packed_name(archive), member.offset_data,\n                                    member.size, row[\'image_sha256\']))\n                seen.add(ident)\n        # Hash remaining TAR padding too, not merely the last image payload.\n        while reader.read(4 * 1024**2):\n            pass\n        if reader.count != proof[\'bytes\'] or reader.digest.hexdigest() != proof[\'sha256\']:\n            raise ValueError(\'Full original TAR SHA mismatch\')\n    if seen != set(expected):\n        raise ValueError(\'Incomplete archive image membership\')\n    return len(seen)\n\n\ndef create_index(root, shards, rows, output):\n    output = Path(output)\n    if output.exists():\n        raise ValueError(\'Preserve an existing index; explicit recovery required\')\n    output.parent.mkdir(parents=True, exist_ok=True)\n    temporary = output.with_suffix(\'.building.sqlite\')\n    groups = {}\n    for row in rows:\n        groups.setdefault(row[\'source_shard\'], []).append(row)\n    if set(groups) != {s[\'file\'] for s in shards}:\n        raise ValueError(\'Archive set differs from manifest\')\n    count = 0\n    with sqlite3.connect(temporary) as db:\n        db.execute(\'PRAGMA synchronous=FULL\')\n        if not db.execute("SELECT name FROM sqlite_master WHERE type=\'table\' AND name=\'images\'").fetchone():\n            db.execute(\'CREATE TABLE images (id TEXT PRIMARY KEY, archive TEXT NOT NULL, offset INTEGER NOT NULL, size INTEGER NOT NULL, sha256 TEXT NOT NULL)\')\n            db.execute(\'CREATE TABLE archives (name TEXT PRIMARY KEY, bytes INTEGER NOT NULL, sha256 TEXT NOT NULL, images INTEGER NOT NULL)\')\n            db.commit()\n        if {x[0] for x in db.execute("SELECT name FROM sqlite_master WHERE type=\'table\'")} != {\'images\', \'archives\'}:\n            raise ValueError(\'Unrecognized incomplete index schema; preserve for inspection\')\n        if db.execute(\'PRAGMA integrity_check\').fetchone()[0] != \'ok\':\n            raise ValueError(\'Incomplete SQLite index failed integrity check\')\n        completed = {name: (size, digest, n) for name, size, digest, n in\n                     db.execute(\'SELECT name,bytes,sha256,images FROM archives\')}\n        if set(completed) - set(groups):\n            raise ValueError(\'Incomplete index contains an unexpected archive\')\n        for shard in shards:\n            name = shard[\'file\']\n            if name in completed:\n                size, digest, n = completed[name]\n                actual = list(db.execute(\'SELECT id,offset,size,sha256 FROM images WHERE archive=?\', (packed_name(name),)))\n                expected = {r[\'id\']: (int(r[\'bytes\']), r[\'image_sha256\']) for r in groups[name]}\n                if (size, digest, n) != (shard[\'bytes\'], shard[\'sha256\'], len(expected)) or len(actual) != n:\n                    raise ValueError(\'Incomplete index archive proof differs from pinned source\')\n                if any(ident not in expected or offset < 0 or (length, image_sha) != expected[ident]\n                       for ident, offset, length, image_sha in actual):\n                    raise ValueError(\'Incomplete index image rows differ from pinned manifest\')\n                path = Path(root) / packed_name(name)\n                if path.stat().st_size != size or sha(path) != digest:\n                    raise ValueError(\'Previously indexed archive bytes changed\')\n                count += n\n                print(json.dumps({\'state\': \'previous_packed_archive_reverified\', \'archive\': name,\n                                  \'verified_images\': count}), flush=True)\n                continue\n            try:\n                db.execute(\'BEGIN\')\n                n = index_archive(root, name, shard, groups[name], db)\n                db.execute(\'INSERT INTO archives VALUES (?,?,?,?)\', (name, shard[\'bytes\'], shard[\'sha256\'], n))\n                db.commit()\n            except BaseException:\n                db.rollback()\n                raise\n            count += n\n            print(json.dumps({\'state\': \'packed_archive_fully_verified\', \'archive\': shard[\'file\'],\n                              \'verified_images\': count}), flush=True)\n        assert db.execute(\'SELECT count(*) FROM images\').fetchone()[0] == count == len(rows)\n    with temporary.open(\'rb\') as stream:\n        os.fsync(stream.fileno())\n    os.replace(temporary, output)\n    return {\'images\': count, \'index_sha256\': sha(output), \'index\': str(output)}\n\n\nclass ArchiveStore:\n    def __init__(self, root, index, index_sha256):\n        self.root, self.source_index = Path(root).resolve(), Path(index).resolve()\n        if sha(self.source_index) != index_sha256:\n            raise ValueError(\'Archive index SHA mismatch\')\n        self.index = self.source_index\n        # Kaggle inputs use a remote filesystem. SQLite\'s tiny random page reads\n        # were the measured bottleneck; cache only this exact 108 MB index.\n        if self.source_index.is_relative_to(Path(\'/kaggle/input\')):\n            cache = Path(\'/kaggle/working/urop_verified_index_cache\')\n            cache.mkdir(parents=True, exist_ok=True)\n            local = cache / (index_sha256 + \'.sqlite\')\n            if not local.exists():\n                temporary = local.with_suffix(\'.tmp-\' + str(os.getpid()))\n                shutil.copyfile(self.source_index, temporary)\n                if sha(temporary) != index_sha256:\n                    raise ValueError(\'Copied index SHA mismatch\')\n                os.replace(temporary, local)\n            if sha(local) != index_sha256:\n                raise ValueError(\'Preserve conflicting cached index\')\n            self.index = local\n        key = (str(self.index), index_sha256)\n        if key not in _INDEX_CACHE:\n            with sqlite3.connect(\'file:\' + str(self.index) + \'?mode=ro&immutable=1\', uri=True) as db:\n                rows = {ident: (name, offset, size, digest) for ident, name, offset, size, digest in\n                        db.execute(\'SELECT id,archive,offset,size,sha256 FROM images\')}\n                proofs = {packed_name(name): (size, digest) for name, size, digest in db.execute(\'SELECT name,bytes,sha256 FROM archives\')}\n            _INDEX_CACHE[key] = rows, proofs\n        self.rows, proofs = _INDEX_CACHE[key]\n        self.sizes = {name: item[0] for name, item in proofs.items()}\n        if self.root.is_relative_to(Path(\'/kaggle/input\')):\n            cache_key = (str(self.root), index_sha256)\n            if cache_key not in _ARCHIVE_CACHE:\n                _ARCHIVE_CACHE[cache_key] = cache_raw_archives(\n                    self.root, proofs, \'/tmp/urop_verified_raw_archives/\' + index_sha256)\n            self.root = _ARCHIVE_CACHE[cache_key]\n        self.pid, self.db, self.handles = None, None, OrderedDict()\n\n    def __getstate__(self):\n        return {**self.__dict__, \'pid\': None, \'db\': None, \'handles\': OrderedDict()}\n\n    def read(self, row):\n        if self.pid != os.getpid():\n            for handle in self.handles.values():\n                handle.close()\n            self.handles, self.pid = OrderedDict(), os.getpid()\n        item = self.rows.get(row[\'id\'])\n        if item is None:\n            raise ValueError(\'Image missing from verified archive index\')\n        name, offset, size, digest = item\n        if name != packed_name(row[\'source_shard\']) or size != int(row[\'bytes\']) or digest != row[\'image_sha256\']:\n            raise ValueError(\'Archive index differs from pinned original manifest\')\n        if name not in self.sizes or offset < 0 or offset + size > self.sizes[name]:\n            raise ValueError(\'Invalid indexed archive byte range\')\n        if name not in self.handles:\n            path = (self.root / name).resolve()\n            if path.parent != self.root or path.stat().st_size != self.sizes[name]:\n                raise ValueError(\'Packed archive path/size changed\')\n            self.handles[name] = path.open(\'rb\', buffering=0)\n            if len(self.handles) > 64:\n                _, old = self.handles.popitem(last=False)\n                old.close()\n        stream = self.handles.pop(name)\n        self.handles[name] = stream\n        stream.seek(offset)\n        raw = stream.read(size)\n        if len(raw) != size or hashlib.sha256(raw).hexdigest() != digest:\n            raise ValueError(\'Indexed image bytes changed\')\n        return raw\n', 'usable_cohort_v2.py': '"""Pinned user-approved available-image cohort; originals remain unchanged.\n\nThis verifies row membership only. It never certifies image bytes or runs models.\n"""\nimport csv\nimport hashlib\nimport json\nfrom collections import Counter\nfrom pathlib import Path\n\nREGISTRATION = \'MM_PART1_USABLE_COHORT_REGISTRATION_20261024.json\'\nREGISTRATION_SHA = \'3fe0b622958a1eee6732789d4f7acf6b450c0e3bda87312e5cbb6500fb405ce5\'\nCOHORT = \'FACULTY-6W-USABLE-682170-v1\'\nPROTOCOL = \'OFFICIAL-6W-MM-USABLE-v2\'\nCOUNTS = {\'train\': 563601, \'validation\': 59299, \'test\': 59270}\nREQUIRED = 682170\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda: f.read(4 * 1024**2), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\n\ndef registration(path):\n    if sha(path) != REGISTRATION_SHA:\n        raise ValueError(\'Approved cohort registration hash mismatch\')\n    value = json.loads(Path(path).read_text())\n    if (value.get(\'approved\') is not True or value.get(\'cohort_id\') != COHORT\n            or value.get(\'protocol_id\') != PROTOCOL or value.get(\'usable_rows\') != REQUIRED\n            or value.get(\'split_counts\') != COUNTS):\n        raise ValueError(\'Wrong approved cohort\')\n    return value\n\n\ndef load_cohort(tsv_dir, registration_path, extraction_path, approval_path):\n    reg = registration(registration_path)\n    if sha(approval_path) != reg[\'approval_sha256\']:\n        raise ValueError(\'User approval receipt mismatch\')\n    if json.loads(Path(approval_path).read_text()).get(\'status\') != \'approved_by_user\':\n        raise ValueError(\'User approval missing\')\n    if sha(extraction_path) != reg[\'extraction_report_sha256\']:\n        raise ValueError(\'Original extraction report hash mismatch\')\n    report = json.loads(Path(extraction_path).read_text())\n    if (report.get(\'source_id\') != reg[\'archive_source_id\']\n            or report.get(\'compressed_archive_sha256\') != reg[\'archive_sha256\']\n            or report.get(\'compressed_archive_bytes\') != reg[\'archive_bytes\']\n            or report.get(\'source_crc_consumed\') is not True\n            or report.get(\'required\') != reg[\'original_rows\']\n            or report.get(\'usable\') != REQUIRED\n            or report.get(\'image_gate_passed\') is not False):\n        raise ValueError(\'Original all-row source evidence differs; never rewrite its failed gate\')\n    failures = report[\'decode_failures\']\n    excluded = {row[\'id\'] for row in failures}\n    fingerprint = hashlib.sha256((\'\\n\'.join(sorted(excluded)) + \'\\n\').encode()).hexdigest()\n    if (len(failures) != len(excluded) or len(excluded) != reg[\'excluded_rows\']\n            or set(report[\'missing_ids\']) != excluded or fingerprint != reg[\'excluded_ids_sha256\']):\n        raise ValueError(\'Exclusions differ from exact approved 491 IDs\')\n    all_ids, found, expected = set(), set(), {}\n    counts, failed_counts, failed_classes = Counter(), Counter(), Counter()\n    classes = {key: Counter() for key in COUNTS}\n    for item in reg[\'original_files\']:\n        path = Path(tsv_dir) / item[\'filename\']\n        if sha(path) != item[\'sha256\']:\n            raise ValueError(\'Original faculty TSV hash mismatch: \' + item[\'filename\'])\n        count = 0\n        with path.open(newline=\'\') as stream:\n            for row in csv.DictReader(stream, delimiter=\'\\t\'):\n                ident = row[\'id\']; label = int(row[\'6_way_label\']); count += 1\n                if ident in all_ids or not ident.isalnum() or label not in range(6) or not row[\'clean_title\'].strip():\n                    raise ValueError(\'Invalid/duplicate original row\')\n                all_ids.add(ident)\n                split = item[\'split\']\n                if ident in excluded:\n                    found.add(ident); failed_counts[split] += 1; failed_classes[str(label)] += 1\n                else:\n                    expected[ident] = {\'id\': ident, \'split\': split,\n                                       \'clean_title\': row[\'clean_title\'], \'6_way_label\': label}\n                    counts[split] += 1; classes[split][str(label)] += 1\n        if count != item[\'rows\']:\n            raise ValueError(\'Original row count mismatch\')\n    if (len(all_ids) != reg[\'original_rows\'] or found != excluded or len(expected) != REQUIRED\n            or dict(counts) != COUNTS or dict(failed_counts) != reg[\'excluded_split_counts\']\n            or dict(failed_classes) != reg[\'excluded_class_counts\']\n            or {key: dict(value) for key, value in classes.items()} != reg[\'class_counts\']):\n        raise ValueError(\'Approved cohort membership/count/label audit failed\')\n    return expected, reg\n\n\ndef load_input(root):\n    root = Path(root)\n    reg = registration(root / \'results\' / REGISTRATION)\n    return load_cohort(root / \'data/official_multimodal_v2\', root / \'results\' / REGISTRATION,\n                       root / \'results\' / reg[\'extraction_report_file\'],\n                       root / \'results\' / reg[\'approval_file\'])\n', 'verify_frozen_image_pair_v1.py': '"""Independent readback of prediction CSVs and selected weight hashes."""\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom packed_image_io_v3 import sha\nfrom verify_multimodal_final_v1 import verify\n\nARMS = (\'vit\', \'resnet50\')\nSCORES = (\'accuracy\', \'macro_f1\', \'balanced_accuracy\')\n\n\ndef read(path):\n    return json.loads(path.read_text())\n\n\ndef verify_run(folder):\n    folder = Path(folder)\n    resolved_path = folder / \'resolved.json\'\n    resolved = read(resolved_path)\n    freeze_path = folder / \'selection_frozen.json\'\n    freeze = read(freeze_path)\n    assert freeze[\'resolved_sha256\'] == sha(resolved_path)\n    assert freeze[\'backbones\'] == resolved[\'backbones\']\n    assert freeze[\'test_manifest_sha256\'] == resolved[\'manifests\'][\'test\']\n    manifests = {}\n    for split, digest in resolved[\'manifests\'].items():\n        path = folder / f\'{split}_manifest.csv\'\n        assert sha(path) == digest\n        frame = pd.read_csv(path, dtype={\'id\': str})\n        assert len(frame) == resolved[\'scope\'][\'counts\'][split]\n        assert frame.id.is_unique and frame.split.eq(split).all()\n        assert set(frame[\'6_way_label\']) == set(range(6))\n        manifests[split] = frame\n    assert pd.concat(list(manifests.values())).id.is_unique\n    test = manifests[\'test\']\n    mask = ~test.image_sha256.isin(set(manifests[\'train\'].image_sha256))\n    subset = test.loc[mask].reset_index(drop=True)\n    if resolved[\'scope\'][\'mode\'] == \'full\':\n        assert len(subset) == 56052\n\n    arms = {}\n    cols = [\'id\', \'true_label\', \'predicted_label\', \'cross_entropy\'] + [f\'p{i}\' for i in range(6)]\n    for arm in ARMS:\n        d = folder / arm\n        history = read(d / \'history.json\')\n        assert [row[\'epoch\'] for row in history] == list(range(1, resolved[\'config\'][\'epochs\'] + 1))\n        epochs = {}\n        for row in history:\n            epoch = row[\'epoch\']\n            proof = verify(d, f\'validation_epoch{epoch}\', manifests[\'validation\'], \'validation\')\n            assert all(abs(proof[\'metrics\'][k] - v) < 1e-10 for k, v in row[\'validation\'].items())\n            assert row[\'weight_sha256\'] == sha(d / f\'epoch{epoch}.pt\')\n            epochs[str(epoch)] = proof\n        best = max(history, key=lambda r: (r[\'validation\'][\'macro_f1\'], -r[\'validation\'][\'loss\'], -r[\'epoch\']))\n        assert freeze[\'selected\'][arm] == best and best[\'weight_sha256\'] == sha(d / \'selected.pt\')\n        saved = read(d / \'test_saved.json\')\n        assert saved[\'freeze_sha256\'] == sha(freeze_path)\n        test_result = verify(d, \'test\', test, \'test\')\n        assert saved[\'test_predictions_sha256\'] == test_result[\'prediction_sha256\']\n        overlap_result = verify(d, \'test_no_train_hash\', subset, \'test\')\n        full_csv = pd.read_csv(d / \'test_predictions.csv\', dtype={\'id\': str}, usecols=cols)\n        sub_csv = pd.read_csv(d / \'test_no_train_hash_predictions.csv\', dtype={\'id\': str}, usecols=cols)\n        assert full_csv.loc[mask, cols].reset_index(drop=True).equals(sub_csv[cols])\n        arms[arm] = {\'epoch\': best[\'epoch\'], \'checkpoint_sha256\': best[\'weight_sha256\'],\n                     \'validation\': epochs, \'test\': test_result, \'test_no_train_hash\': overlap_result}\n    delta = {name: {k: arms[\'vit\'][name][\'metrics\'][k] - arms[\'resnet50\'][name][\'metrics\'][k]\n                    for k in SCORES} for name in (\'test\', \'test_no_train_hash\')}\n    return {\'status\': \'passed\', \'scope\': resolved[\'scope\'], \'test_posts\': len(test),\n            \'no_train_hash_posts\': len(subset), \'arms\': arms, \'differences\': delta,\n            \'frozen_selection_sha256\': sha(freeze_path), \'resolved_sha256\': sha(resolved_path)}\n\n\ndef report(folder, proof):\n    scope = proof[\'scope\']\n    resolved = read(folder / \'resolved.json\')\n    lines = [\'# Frozen ViT-B/16 versus ResNet-50\', \'\', f"Scope: {scope[\'mode\']}; "\n             + \', \'.join(f"{split}: {count:,}" for split, count in scope[\'counts\'].items()) + \'.\',\n             f"Latest run session: {read(folder / \'runtime.json\')[\'current_session_seconds\']/60:.1f} minutes.",\n             \'\', \'| Held-out split | Model | Accuracy | Macro-F1 | Balanced accuracy |\',\n             \'|---|---|---:|---:|---:|\']\n    for key, name, n in [(\'test\', \'All selected test posts\', proof[\'test_posts\']),\n                          (\'test_no_train_hash\', \'Exact bytes absent from training\', proof[\'no_train_hash_posts\'])]:\n        for arm, info in proof[\'arms\'].items():\n            m = info[key][\'metrics\']\n            lines.append(f"| {name} ({n:,}) | {arm} | {m[\'accuracy\']:.4%} | {m[\'macro_f1\']:.6f} | {m[\'balanced_accuracy\']:.4%} |")\n        m = proof[\'differences\'][key]\n        lines.append(f"| {name} ({n:,}) | ViT − ResNet-50 | {100*m[\'accuracy\']:+.4f} pp | {m[\'macro_f1\']:+.6f} | {100*m[\'balanced_accuracy\']:+.4f} pp |")\n    for arm in ARMS:\n        lines += [\'\', f"## {arm} (epoch {proof[\'arms\'][arm][\'epoch\']})", \'\',\n                  \'| Test cohort | Class | Precision | Recall | F1 | Support |\',\n                  \'|---|---|---:|---:|---:|---:|\']\n        for prefix in (\'test\', \'test_no_train_hash\'):\n            items = read(folder / arm / f\'{prefix}_classification_report.json\')\n            for label, m in items.items():\n                if isinstance(m, dict) and label not in (\'macro avg\', \'weighted avg\', \'micro avg\'):\n                    lines.append(f"| {prefix} | {label} | {m[\'precision\']:.6f} | {m[\'recall\']:.6f} | {m[\'f1-score\']:.6f} | {int(m[\'support\']):,} |")\n    lines += [\'\', \'Exact image hashes identify byte-for-byte repeats only. The main comparison is against the matched frozen ResNet-50.\',\n              \'The historical fine-tuned ResNet-50 scored 81.1557% accuracy and 0.661796 macro-F1 under its original protocol.\',\n              \'The backbone pretraining recipes differ. This single-seed result does not establish run-to-run variability.\',\n              f"Runtime hardware: {\', \'.join(resolved[\'runtime\'][\'devices\'])}.", \'\']\n    return \'\\n\'.join(lines)\n\n\nif __name__ == \'__main__\':\n    import argparse\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'folder\', type=Path)\n    args = parser.parse_args()\n    result = verify_run(args.folder)\n    receipt = args.folder / \'independent_verification.json\'\n    if receipt.is_file():\n        assert result == read(receipt), \'Saved verification receipt changed\'\n    print(json.dumps(result, indent=2))\n', 'verify_multimodal_final_v1.py': '"""Independent CSV-only verification for a frozen multimodal validation/test.\nNo model, inference, training or network access. Output is a returned dict.\n"""\nfrom pathlib import Path\nimport hashlib,json\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import accuracy_score,f1_score,balanced_accuracy_score,classification_report,confusion_matrix\nLABELS=[\'True\',\'Satire/Parody\',\'Misleading Content\',\'Imposter Content\',\'False Connection\',\'Manipulated Content\']\ndef sha(path):\n h=hashlib.sha256()\n with Path(path).open(\'rb\') as f:\n  for b in iter(lambda:f.read(4*1024**2),b\'\'):h.update(b)\n return h.hexdigest()\ndef verify(folder,prefix,official,split):\n folder=Path(folder);frame=pd.read_csv(folder/(prefix+\'_predictions.csv\'),dtype={\'id\':str})\n assert len(frame)==len(official) and frame.id.is_unique and frame.id.tolist()==official.id.tolist()\n assert frame[\'split\'].eq(split).all()\n y=frame.true_label.to_numpy();p=frame.predicted_label.to_numpy();ce=frame.cross_entropy.to_numpy();probs=frame[[\'p\'+str(i) for i in range(6)]].to_numpy()\n assert np.array_equal(y,official[\'6_way_label\'].astype(int).to_numpy())\n assert np.isfinite(probs).all() and np.isfinite(ce).all() and (ce>=0).all() and (probs>=0).all() and (probs<=1).all()\n assert np.allclose(probs.sum(1),1,atol=2e-6,rtol=0) and np.array_equal(probs.argmax(1),p)\n gold=probs[np.arange(len(y)),y];positive=gold>0\n assert np.allclose(-np.log(gold[positive]),ce[positive],rtol=2e-5,atol=2e-5)\n assert (ce[~positive]>=-np.log(np.finfo(np.float32).tiny)).all()\n metrics={\'accuracy\':float(accuracy_score(y,p)),\'micro_f1\':float(f1_score(y,p,average=\'micro\')),\'macro_f1\':float(f1_score(y,p,average=\'macro\',labels=range(6),zero_division=0)),\'weighted_f1\':float(f1_score(y,p,average=\'weighted\',labels=range(6),zero_division=0)),\'balanced_accuracy\':float(balanced_accuracy_score(y,p)),\'loss\':float(ce.mean()),\'samples\':len(frame)}\n def compare(a,b):\n  if isinstance(a,dict):\n   assert set(a)==set(b)\n   for k in a:compare(a[k],b[k])\n  else:assert abs(a-b)<1e-10\n compare(json.loads((folder/(prefix+\'_metrics.json\')).read_text()),metrics)\n compare(json.loads((folder/(prefix+\'_classification_report.json\')).read_text()),classification_report(y,p,labels=range(6),target_names=LABELS,output_dict=True,zero_division=0))\n cm=confusion_matrix(y,p,labels=range(6));conf=json.loads((folder/(prefix+\'_confusions.json\')).read_text());assert conf[\'labels\']==LABELS and np.array_equal(conf[\'raw\'],cm)\n norm=np.divide(cm,cm.sum(1,keepdims=True),out=np.zeros_like(cm,dtype=float),where=cm.sum(1,keepdims=True)!=0)\n assert np.allclose(conf[\'row_normalized\'],norm,atol=1e-12,rtol=0)\n return {\'status\':\'passed_independent_csv_verification\',\'split\':split,\'metrics\':metrics,\'prediction_sha256\':sha(folder/(prefix+\'_predictions.csv\')),\'exact_order_ids_labels\':True,\'probability_ce_confusion_checks\':True}\n'}
PROVENANCE = {'checkout_head': '0409f7d031734cddd9d19052ff570637dead42f7', 'embedded_source_sha256': {'frozen_image_pair_v1.py': '81a324a2e358f5d9e6737db438ef29983cf0f39ef49da4f4c370881be0e6a6f1', 'image_only_vit_resnet_frozen_v1.json': 'a7c9c28989952fcf51208260b1d770765d9773e6d5a2d3fd0e4ce2d362f6c319', 'packed_image_io_v3.py': 'a29e66d9c2962eb5bdfdce9abab02866fc77ad44aa5711ec8009a09dda7bb60d', 'usable_cohort_v2.py': '22bff1eb0fa14d102ad07786f936c5c1ed1582ec7bd5cee2b8e8dc5eb3520092', 'verify_frozen_image_pair_v1.py': '8655504e6940be8ead41659eef7e0f2bd2ba0658a7606c2404eec3ba812eecd0', 'verify_multimodal_final_v1.py': '2167e194447eacbcfd2c45cd70f07c524e8996b87fd1e702a9b4900de615065b'}}


In [ ]:
import sys, json, shutil, hashlib
from pathlib import Path
source = Path('/tmp/urop_frozen_pair_source')
source.mkdir(exist_ok=True)
for name, body in SOURCES.items():
    assert hashlib.sha256(body.encode()).hexdigest() == PROVENANCE['embedded_source_sha256'][name]
    (source/name).write_text(body)
sys.path.insert(0, str(source))
from frozen_image_pair_v1 import run
out = Path('/kaggle/working/urop_frozen_pair')
if RECOVERY is not None and not out.exists():
    shutil.copytree(RECOVERY, out)
cfg = json.loads((source/'image_only_vit_resnet_frozen_v1.json').read_text())
run(Path('/kaggle/input'), out, cfg, PROVENANCE, GPU_HOURS_OBSERVED, SESSION_SECONDS, MODE)
